In [12]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.metrics import classification_report, confusion_matrix

In [13]:
# Ruta al root del repo (ajusta si hace falta)
PROJECT_ROOT = Path.cwd().parents[1]

# Rutas a datos y modelo
DATA_PATH = PROJECT_ROOT / "ml" / "data" / "model2_seniority" / "3.processed" / "cv_with_seniority_weak.csv"
MODEL_PATH = PROJECT_ROOT / "ml" / "models" / "seniority" / "seniority_from_cv_balanced_v1.pkl"

print("DATA_PATH:", DATA_PATH)
print("MODEL_PATH:", MODEL_PATH)

# 1) Cargar datos y modelo
df = pd.read_csv(DATA_PATH)

DATA_PATH: c:\Users\Fiona A\Desktop\IAPython\proyectos\TechCareer\TechCareer\backend\ml\data\model2_seniority\3.processed\cv_with_seniority_weak.csv
MODEL_PATH: c:\Users\Fiona A\Desktop\IAPython\proyectos\TechCareer\TechCareer\backend\ml\models\seniority\seniority_from_cv_balanced_v1.pkl


In [14]:
seniority_model = joblib.load(MODEL_PATH)


In [15]:
# 2) Filtrar a filas con etiqueta válida
mask = df["seniority_weak"].isin(["Junior", "Mid", "Senior"])
df_sen = df[mask].copy()

print("Filas con etiqueta válida:", df_sen.shape[0])
print(df_sen["seniority_weak"].value_counts())

# 3) Tomar una muestra para evaluar (por ejemplo 500 CVs)
df_sample = df_sen.sample(n=500, random_state=42)

X_sample = df_sample["cv_text"]
y_true = df_sample["seniority_weak"]

# 4) Predicciones del modelo guardado
y_pred = seniority_model.predict(X_sample)

print("=== Classification report (modelo guardado, muestra 500) ===")
print(classification_report(y_true, y_pred, labels=["Junior", "Mid", "Senior"]))

print("=== Matriz de confusión ===")
print(confusion_matrix(y_true, y_pred, labels=["Junior", "Mid", "Senior"]))

Filas con etiqueta válida: 5253
Senior    4594
Mid        405
Junior     254
Name: seniority_weak, dtype: int64
=== Classification report (modelo guardado, muestra 500) ===
              precision    recall  f1-score   support

      Junior       0.91      0.95      0.93        21
         Mid       0.78      0.92      0.84        38
      Senior       0.99      0.97      0.98       441

    accuracy                           0.97       500
   macro avg       0.89      0.95      0.92       500
weighted avg       0.97      0.97      0.97       500

=== Matriz de confusión ===
[[ 20   0   1]
 [  0  35   3]
 [  2  10 429]]


In [19]:
file_path = "new_cvs_for_seniority_test.csv"  # ajusta al nombre real
df_new = pd.read_csv(file_path)

In [20]:
df_new["seniority_pred"] = seniority_model.predict(df_new["cv_text"])

In [21]:
print(df_new[["cv_id", "seniority_manual", "seniority_pred", "role_label_manual"]])

   cv_id seniority_manual seniority_pred          role_label_manual
0      1           Junior         Junior             data_scientist
1      2           Junior         Junior          backend_developer
2      3              Mid            Mid              data_engineer
3      4              Mid         Junior               data_analyst
4      5              Mid            Mid          backend_developer
5      6           Senior         Senior  machine_learning_engineer
6      7           Senior            Mid             data_scientist
7      8           Senior         Senior              data_engineer
8      9           Senior            Mid            other_data_role
9     10           Junior         Junior  machine_learning_engineer


In [22]:
from sklearn.metrics import classification_report

print(classification_report(df_new["seniority_manual"], df_new["seniority_pred"], labels=["Junior","Mid","Senior"]))


              precision    recall  f1-score   support

      Junior       0.75      1.00      0.86         3
         Mid       0.50      0.67      0.57         3
      Senior       1.00      0.50      0.67         4

    accuracy                           0.70        10
   macro avg       0.75      0.72      0.70        10
weighted avg       0.78      0.70      0.70        10



In [23]:
probs = seniority_model.predict_proba(df_new["cv_text"])
print(seniority_model.classes_)
print(probs)


['Junior' 'Mid' 'Senior']
[[0.76312704 0.16471887 0.07215409]
 [0.50829229 0.29713405 0.19457366]
 [0.38938568 0.40852514 0.20208918]
 [0.47084072 0.3897463  0.13941299]
 [0.14940821 0.44810345 0.40248833]
 [0.21543902 0.25181222 0.53274876]
 [0.23820891 0.39846063 0.36333046]
 [0.16169335 0.27953882 0.55876784]
 [0.29056344 0.37287766 0.3365589 ]
 [0.75766893 0.13936323 0.10296783]]
